### 3.6.4. Mixed-Integer Quadratic Programming (MIQP)

$$
\begin{aligned}
\min_{\mathbf{x}}\quad & \tfrac12\mathbf{x}^\top Q \mathbf{x} + \mathbf{c}^\top \mathbf{x} \\
\text{subject to}\quad & A\mathbf{x} \le \mathbf{b},\\
& x_i \in \mathbb{Z}\ \text{for } i\in\mathcal{I},
\end{aligned}
$$
with $Q\succeq 0$ for a convex MIQP.

**Explanation:**

A mixed-integer quadratic program (MIQP) keeps a convex quadratic objective but forces some variables to be integer. Even when $Q\succeq 0$ makes each relaxed subproblem a convex [QP](../04_Convex_Problem_Classes/02_quadratic_programming.ipynb), the integrality requirement makes the overall problem combinatorial. Geometrically the objective is a paraboloid and the solution is the integer feasible point closest (in the $Q$-weighted metric) to the unconstrained minimizer, found by branch-and-bound over the integer variables.

**Numerical Example:**

$$
\min\; (x_1 - 2.3)^2 + (x_2 - 0.8)^2
\quad\text{s.t.}\quad x_1 + x_2 \le 3,\; x_1,x_2\in\mathbb{Z}_{\ge 0}.
$$

The continuous minimizer is $(2.3, 0.8)$. Rounding each coordinate to the nearest feasible integers and comparing:
$$
(2,1):\ 0.09 + 0.04 = 0.13,\quad
(2,0):\ 0.09 + 0.64 = 0.73,\quad
(3,0):\ 0.49 + 0.64 = 1.13.
$$
The optimum is $\mathbf{x}^\star = (2,1)$ with value $0.13$ — the nearest lattice point that still satisfies $x_1+x_2\le 3$.

In [1]:
target = (2.3, 0.8)

def objective(x_1, x_2):
    return (x_1 - target[0])**2 + (x_2 - target[1])**2

feasible_points = [(x_1, x_2)
                   for x_1 in range(0, 4)
                   for x_2 in range(0, 4)
                   if x_1 + x_2 <= 3]
best_point = min(feasible_points, key=lambda point: objective(*point))

print("candidate objective values:",
      {point: round(objective(*point), 2) for point in [(2, 1), (2, 0), (3, 0)]})
print("integer optimum:", best_point, "value:", round(objective(*best_point), 2))

candidate objective values: {(2, 1): 0.13, (2, 0): 0.73, (3, 0): 1.13}
integer optimum: (2, 1) value: 0.13


**Equivalent casadi (bonmin) implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = (decision_variable[0] - 2.3)**2 + (decision_variable[1] - 0.8)**2
constraint = decision_variable[0] + decision_variable[1]

solver = ca.nlpsol("solver", "bonmin",
                   {"x": decision_variable, "f": objective, "g": constraint},
                   {"discrete": [True, True], "bonmin.bb_log_level": 0, "print_time": 0})
solution = solver(x0=[0, 0], lbg=-ca.inf, ubg=3, lbx=0, ubx=3)

print("integer optimum:", solution["x"], " value:", round(float(solution["f"]), 2))


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.004999997        9 0.001802
NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.12999997        5 0.000854
NLP0014I             2         OPT 0.13        0 0
NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.13        0 0
NLP0014I             2         OPT 0.089999988        5 0.000973
NLP0014I         

**References:**

[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Mixed-Integer Linear Programming (MILP)](./03_mixed_integer_linear_programming.ipynb) | [Next: Mixed-Integer Nonlinear Programming (MINLP) ➡️](./05_mixed_integer_nonlinear_programming.ipynb)